# Introduction

In [34]:
!sed -n '670,700p' /content/openwakeword/openwakeword/train.py


        # Generate positive clips for training
        logging.info("#"*50 + "\nGenerating positive clips for training\n" + "#"*50)
        if not os.path.exists(positive_train_output_dir):
            os.mkdir(positive_train_output_dir)
        n_current_samples = len(os.listdir(positive_train_output_dir))
        if n_current_samples <= 0.95*config["n_samples"]:
            generate_samples(
                text=config["target_phrase"], max_samples=config["n_samples"]-n_current_samples,
                batch_size=config["tts_batch_size"],
                noise_scales=[0.98], noise_scale_ws=[0.98], length_scales=[0.75, 1.0, 1.25],
                output_dir=positive_train_output_dir, auto_reduce_batch_size=True,
                file_names=[uuid.uuid4().hex + ".wav" for i in range(config["n_samples"])]
            )
            torch.cuda.empty_cache()
        else:
            logging.warning(f"Skipping generation of positive clips for training, as ~{config['n_samples']} already exist

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [2]:
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt
'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 184 (delta 70), reused 53 (delta 53), pack-reused 98 (from 1)
Receiving objects: 100% (184/184), 1.04 MiB | 22.69 MiB/s, done.
Resolving deltas: 100% (93/93), done.
wget: missing URL
Usage: wget [OPTION]... [URL]...

Try `wget --help' for more options.
ERROR: Could not find a version that satisfies the requirement piper-phonemize (from versions: none)
ERROR: No matching distribution found for piper-phonemize
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for webrtcvad: filename=webrtcvad-2.0.10-cp312-cp312-linux_x86_64.whl size=73514 sha256=af777fa032ec6f1790e02f8f4be31ea4b7b7d5804d65e83a058c380d2334e06f
  Stored in directory: /root/.cache/pip/wheels/1e/d3/95/680fa3b16848f1a58d2edaed34c496224c89a9bc63e17b3614
S

In [3]:
!pip install piper-tts==1.3.0
!rm -rf piper-sample-generator
!git clone --branch v3.0.0 https://github.com/rhasspy/piper-sample-generator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 78.0 MB/s eta 0:00:00
Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 184 (delta 70), reused 53 (delta 53), pack-reused 98 (from 1)
Receiving objects: 100% (184/184), 1.04 MiB | 16.93 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Note: switching to '4d7e4b390c29bac54dd83e5f6688ef39ab12d578'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedH

In [4]:
import urllib.request
url = "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"
urllib.request.urlretrieve(url, "piper-sample-generator/models/en_US-libritts_r-medium.pt")

('piper-sample-generator/models/en_US-libritts_r-medium.pt',
 <http.client.HTTPMessage at 0x7e533cbbc7d0>)

In [6]:
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)

In [7]:
import urllib.request
base = "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/"
dest = "./openwakeword/openwakeword/resources/models/"
urllib.request.urlretrieve(base + "embedding_model.onnx", dest + "embedding_model.onnx")

('./openwakeword/openwakeword/resources/models/embedding_model.onnx',
 <http.client.HTTPMessage at 0x7e533cd03650>)

In [8]:
urllib.request.urlretrieve(base + "embedding_model.tflite", dest + "embedding_model.tflite")

('./openwakeword/openwakeword/resources/models/embedding_model.tflite',
 <http.client.HTTPMessage at 0x7e533cd03a70>)

In [9]:
urllib.request.urlretrieve(base + "melspectrogram.onnx", dest + "melspectrogram.onnx")

('./openwakeword/openwakeword/resources/models/melspectrogram.onnx',
 <http.client.HTTPMessage at 0x7e533cfea060>)

In [10]:
urllib.request.urlretrieve(base + "melspectrogram.tflite", dest + "melspectrogram.tflite")

('./openwakeword/openwakeword/resources/models/melspectrogram.tflite',
 <http.client.HTTPMessage at 0x7e533cfea480>)

In [15]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [16]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

270it [03:36,  1.25it/s]


In [17]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Convert audioset files to 16khz sample rate
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset (https://github.com/mdeff/fma)
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break


--2026-06-24 22:05:37--  https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
Resolving huggingface.co (huggingface.co)... 13.35.202.121, 13.35.202.34, 13.35.202.97, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.121|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-06-24 22:05:37 ERROR 404: Not Found.

tar: This does not look like a tar archive
tar: Exiting with failure status due to previous errors


0it [00:00, ?it/s]


 99%|█████████▉| 119/120 [00:37<00:00,  3.17it/s]


In [19]:
import os
print([f for f in os.listdir(".") if "ACAV100M" in f or "validation_set" in f])

['validation_set_features.npy', 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy.1', 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy', 'validation_set_features.npy.1']


In [20]:
import os
os.remove("openwakeword_features_ACAV100M_2000_hrs_16bit.npy.1")
os.remove("validation_set_features.npy.1")

In [21]:
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

config["target_phrase"] = ["hey atom"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25
config["background_paths"] = ["./fma"]
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
import yaml as _yaml; _yaml.dump(config, open('my_model.yaml', 'w'))
print(config)

{'model_name': 'hey_atom', 'target_phrase': ['hey atom'], 'custom_negative_phrases': [], 'n_samples': 1000, 'n_samples_val': 1000, 'tts_batch_size': 50, 'augmentation_batch_size': 16, 'piper_sample_generator_path': './piper-sample-generator', 'output_dir': './my_custom_model', 'rir_paths': ['./mit_rirs'], 'background_paths': ['./fma'], 'background_paths_duplication_rate': [1], 'false_positive_validation_data_path': 'validation_set_features.npy', 'augmentation_rounds': 1, 'feature_data_files': {'ACAV100M_sample': 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'}, 'batch_n_per_class': {'ACAV100M_sample': 1024, 'adversarial_negative': 50, 'positive': 50}, 'model_type': 'dnn', 'layer_size': 32, 'steps': 10000, 'max_negative_weight': 1500, 'target_false_positives_per_hour': 0.2, 'target_accuracy': 0.6, 'target_recall': 0.25}


In [18]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

--2026-06-24 22:07:39--  https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
Resolving huggingface.co (huggingface.co)... 13.35.202.34, 13.35.202.40, 13.35.202.97, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.34|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/64f3a0b6918ffcc15af6923c/7e1cade4c3fda6a5081158383c8d43c4a3e1e42555150b596b373efddf9b5194?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27openwakeword_features_ACAV100M_2000_hrs_16bit.npy%3B+filename%3D%22openwakeword_features_ACAV100M_2000_hrs_16bit.npy%22%3B&user_id=public&Expires=1782342459&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRmM2EwYjY5MThmZmNjMTVhZjY5MjNjLzdlMWNhZGU0YzNmZGE2YTUwODExNTgzODNjOGQ0M2M0YTNlMWU0MjU1NTE1MGI1OTZiMzczZWZkZGY5YjUxOTRcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZy

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
config["target_phrase"] = ["hey atom"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25
config["background_paths"] = ["./fma"]
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
import yaml as _yaml; _yaml.dump(config, open('my_model.yaml', 'w'))
print(config)

In [14]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

NameError: name 'yaml' is not defined

# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [25]:
path = "/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py"
content = open(path).read()
content = content.replace('torchaudio.set_audio_backend("soundfile")', 'pass')
open(path, "w").write(content)

8856

In [27]:
!rm -rf piper-sample-generator
!git clone --branch v3.0.0 https://github.com/rhasspy/piper-sample-generator

Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 184 (delta 70), reused 53 (delta 53), pack-reused 98 (from 1)
Receiving objects: 100% (184/184), 1.04 MiB | 6.75 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Note: switching to '4d7e4b390c29bac54dd83e5f6688ef39ab12d578'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



In [31]:
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt
url = "https://github.com/rhasspy"
url += "/piper-sample-generator"
url += "/releases/download/v2.0.0"
url += "/en_US-libritts_r-medium.pt"

wget: missing URL
Usage: wget [OPTION]... [URL]...

Try `wget --help' for more options.


In [32]:
import urllib.request
urllib.request.urlretrieve(url, "piper-sample-generator/models/en_US-libritts_r-medium.pt")

('piper-sample-generator/models/en_US-libritts_r-medium.pt',
 <http.client.HTTPMessage at 0x7a2a74479190>)

In [22]:
path3 = "/content/openwakeword/openwakeword/train.py"

In [23]:
content3 = open(path3).read()

In [24]:
content3 = content3.replace("generate_samples(",'generate_samples(model="piper-sample-generator/models/en_US-libritts_r-medium.pt", ')

In [25]:
open(path3, "w").write(content3)

45644

In [53]:
# Step 1: Generate synthetic clips
# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

INFO:root:##################################################
Generating positive clips for training
##################################################
DEBUG:generate_samples:Loading piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:generate_samples:Successfully loaded the model
DEBUG:generate_samples:CUDA available, using GPU
DEBUG:generate_samples:Batch 1/40 complete
DEBUG:generate_samples:Batch 2/40 complete
DEBUG:generate_samples:Batch 3/40 complete
DEBUG:generate_samples:Batch 4/40 complete
DEBUG:generate_samples:Batch 5/40 complete
DEBUG:generate_samples:Batch 6/40 complete
DEBUG:generate_samples:Batch 7/40 complete
DEBUG:generate_samples:Batch 8/40 complete
DEBUG:generate_samples:Batch 9/40 complete
DEBUG:generate_samples:Batch 10/40 complete
DEBUG:generate_samples:Batch 11/40 complete
DEBUG:generate_samples:Batch 12/40 complete
DEBUG:generate_samples:Batch 13/40 complete
DEBUG:generate_samples:Batch 14/40 complete
DEBUG:generate_samples:Batch 15/40 complete
DEBUG:gen

In [54]:
import soundfile as sf

In [55]:
import librosa

In [56]:
import glob

In [57]:
files = glob.glob("my_custom_model/hey_atom/*/*.wav")

In [58]:
len(files)

8000

In [59]:
def resample_if_needed(fpath): d, sr = sf.read(fpath); return None if sr == 16000 else sf.write(fpath,librosa.resample(d, orig_sr=sr, target_sr=16000), 16000)
[resample_if_needed(f) for f in files]

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [49]:
import os
print(os.listdir("my_custom_model"))

['hey_atom']


In [50]:
import os
print(os.listdir("my_custom_model/hey_atom"))

['positive_features_train.npy', 'negative_test', 'positive_train', 'positive_test', 'negative_train']


In [52]:
import soundfile as sf
import os
sample_file = "my_custom_model/hey_atom/positive_train/" + os.listdir("my_custom_model/hey_atom/positive_train")[0]
print(sf.info(sample_file))

my_custom_model/hey_atom/positive_train/1140d6e6a68b497e8b73dcdc6cd85c01.wav
samplerate: 22050 Hz
channels: 1
duration: 19712 samples
format: WAV (Microsoft) [WAV]
subtype: Signed 16 bit PCM [PCM_16]


In [54]:
import soundfile as sf

In [55]:
import librosa

In [56]:
import glob

In [57]:
files = glob.glob("my_custom_model/hey_atom/*/*.wav")

In [44]:
def resample_if_needed(fpath): d, sr = sf.read(fpath); return None if sr == 16000 else sf.write(fpath,
librosa.resample(d, orig_sr=sr, target_sr=16000), 16000)
[resample_if_needed(f) for f in files]

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [45]:
import glob, os
[os.remove(f) for f in glob.glob("my_custom_model/hey_atom/*.npy")]

[]

In [61]:
import os
os.remove("my_custom_model/hey_atom/positive_features_train.npy")

In [63]:
path4 = "/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py"

In [64]:
content4 = open(path4).read()

In [65]:
old_line = "info = torchaudio.info(str(file_path))"

In [68]:
new_line = "info = __import__('soundfile').info(str(file_path)); info.num_frames = info.frames; info.sample_rate = info.samplerate"

In [69]:
content4 = content4.replace(old_line, new_line)

In [70]:
open(path4, "w").write(content4)

8856

In [72]:
"soundfile" in open("/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py").read()

False

In [73]:
!grep -n "torchaudio.info" /usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py

101:        info = torchaudio.info(file_path)


In [74]:
import glob
[os.remove(f) for f in glob.glob("my_custom_model/hey_atom/*.npy")]

[None]

In [89]:
path5 = "/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py"

In [90]:
content5 = open(path5).read()
content5 = content5.replace("info = torchaudio.info(file_path)", "info = __import__('soundfile').info(str(file_path));info.num_frames = info.frames; info.sample_rate = info.samplerate")

In [91]:
open(path5, "w").write(content5)

0

In [92]:
!pip install --force-reinstall --no-deps torch-audiomentations==0.11.0

  Using cached torch_audiomentations-0.11.0-py3-none-any.whl.metadata (12 kB)
Using cached torch_audiomentations-0.11.0-py3-none-any.whl (47 kB)
  Attempting uninstall: torch-audiomentations
    Found existing installation: torch-audiomentations 0.11.0
    Uninstalling torch-audiomentations-0.11.0:
      Successfully uninstalled torch-audiomentations-0.11.0


In [94]:
len(open("/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py").read())

8893

In [26]:
path6 = "/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py"

In [27]:
content6 = open(path6).read()

In [28]:
content6 = content6.replace('torchaudio.set_audio_backend("soundfile")', 'pass')

In [29]:
content6 = content6.replace("info = torchaudio.info(file_path)", "info = __import__('soundfile').info(str(file_path));info.num_frames = info.frames; info.sample_rate = info.samplerate")

In [30]:
len(content6)

8940

In [31]:
open(path6, "w").write(content6)

8940

In [32]:
"soundfile" in open(path6).read()

True

In [33]:
import glob, os
[os.remove(f) for f in glob.glob("my_custom_model/hey_atom/*.npy")]

[]

In [2]:
import sys
print(sys.executable)
print('config' in dir())

/usr/bin/python3
False


In [1]:
import os
print(os.path.exists("my_model.yaml"))
print(os.path.exists("my_custom_model/hey_atom"))
print("soundfile" in open("/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py").read())

FileNotFoundError: [Errno 2] No such file or directory: '/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py'

In [60]:
# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

INFO:root:##################################################
Computing openwakeword features for generated samples
##################################################
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_audiomentations/core/transforms_interface.py:77: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default to 'dict' in v0.12, and will be removed in v0.13. Make sure to update your code to something like:
  >>> augment = PitchShift(..., output_type='dict')
  >>> augmented_samples = augment(samples).samples
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_audiomentations/core/transforms_interface.py:77: FutureWarning: Transforms now expect a

In [48]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 20.3 MB/s eta 0:00:00


In [51]:
config["n_samples"] = 3000
config["steps"] = 30000
config["target_accuracy"] = 0.7
config["target_recall"] = 0.5
import yaml as _yaml; _yaml.dump(config, open('my_model.yaml', 'w'))

In [52]:
import glob, os
[os.remove(f) for f in glob.glob("my_custom_model/hey_atom/*.npy")]

[None, None, None, None]

In [61]:
# Step 3: Train model

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

INFO:root:##################################################
Starting training sequence 1...
##################################################
Training: 100% 29999/30000 [12:29<00:00, 40.01it/s]
INFO:root:##################################################
Starting training sequence 2...
##################################################
INFO:root:Increasing weight on negative examples to reduce false positives...
Training: 100% 2999/3000.0 [03:15<00:00, 15.37it/s]
INFO:root:##################################################
Starting training sequence 3...
##################################################
INFO:root:Increasing weight on negative examples to reduce false positives...
Training: 100% 2999/3000.0 [03:17<00:00, 15.22it/s]
INFO:root:Merging checkpoints above the 90th percentile into single model...
INFO:root:
################
Final Model Accuracy: 0.7860000133514404
Final Model Recall: 0.5740000009536743
Final Model False Positives per Hour: 5.663716793060303
###############

In [1]:
import os
print(os.listdir("my_custom_model"))
print(os.listdir("my_custom_model/hey_atom"))

False


FileNotFoundError: [Errno 2] No such file or directory: 'my_custom_model/hey_atom.onnx'

In [21]:
# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly
# If so, run this cell to retry

# Manually save to tflite as this doesn't work right in colab
def convert_onnx_to_tflite(onnx_model_path, output_path):
    """Converts an ONNX version of an openwakeword model to the Tensorflow tflite format."""
    # imports
    import onnx
    import logging
    import tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf

    # Convert to tflite from onnx model
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()

        logging.info(f"####\nSaving tflite mode to '{output_path}'")
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

    return None

convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")


ModuleNotFoundError: No module named 'onnx'

After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!